# 03 — Machine learning with spatially explicit validation

**RQ1:** Do 1-month-lagged wildfire + remote-sensing disturbance variables improve prediction of H5N1 wild-bird detections beyond climate + seasonality?

| Model | Features | Role |
|---|---|---|
| Logistic regression | climate, seasonality, agriculture | interpretable epidemiologic baseline |
| Random Forest | baseline + fire (count, FRP), NDVI anomaly, LST anomaly | non-linear interactions, ML benchmark |

**Validation:** `GroupKFold` by county — every test fold is a set of *unseen counties*, guarding against spatial leakage. Pooled out-of-fold predictions give honest ROC/PR curves. With rare positives, **PR-AUC is the more informative metric**; bootstrap CIs quantify uncertainty.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src import config
from src.models import spatial_cv
from src import viz

from src.features import load_panel
panel = load_panel()

## 1. Run county-grouped cross-validation

In [2]:
metrics, curves, d = spatial_cv(panel, config.FULL_FEATURES, target=config.TARGET)
metrics.round(3)

,model,roc_auc,pr_auc,precision,recall,f1,n,n_positive
0,Logistic (baseline),0.740,0.406,0.636,0.156,0.250,2784,436
1,Random Forest (full),0.752,0.473,0.502,0.360,0.419,2784,436


## 2. Discrimination: ROC and precision–recall (out-of-fold)

In [3]:
viz.roc_pr_panel(curves, metrics, config.FIGURES/"fig4_model_performance.png")

## 3. Uncertainty: bootstrap CIs on the headline metrics

In [4]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from src.features import build_xy
from src.evaluate import bootstrap_auc_ci

X, y, groups, d2 = build_xy(panel, config.FULL_FEATURES, config.TARGET)
oof = cross_val_predict(RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                               class_weight="balanced_subsample",
                                               n_jobs=-1, random_state=42),
                        X, y, groups=groups, cv=GroupKFold(5), method="predict_proba")[:, 1]
print("ROC-AUC 95% CI:", np.round(bootstrap_auc_ci(y, oof, metric="roc"), 3))
print("PR-AUC  95% CI:", np.round(bootstrap_auc_ci(y, oof, metric="pr"), 3))

ROC-AUC 95% CI: [0.725 0.777]
PR-AUC  95% CI: [0.42  0.522]


## 4. Persist results

In [5]:
metrics.to_csv(config.RESULTS / "cv_metrics.csv", index=False)
print(metrics.round(3))

                  model  roc_auc  pr_auc  precision  recall     f1     n  \
0   Logistic (baseline)    0.740   0.406      0.636   0.156  0.250  2784   
1  Random Forest (full)    0.752   0.473      0.502   0.360  0.419  2784   

   n_positive  
0         436  
1         436  


## Interpretation guide

- **If full-model ROC/PR-AUC clearly exceed the baseline** → disturbance carries usable early-warning signal; inspect which features drive it (notebook 04).
- **If they do not** → consistent with Zou et al. (2026, medRxiv preprint) at coarse scales, and motivates finer mechanisms (smoke plume exposure, telemetry-tracked displacement) rather than county-level acres burned.

Either outcome is reported honestly in the README — a disciplined null is a portfolio asset, not a failure.